# 02 — Simulateur NDJSON IoT Hub

**Objectif** : lire Silver Delta et produire des fichiers NDJSON
qui imitent exactement ce qu'Azure IoT Hub écrirait dans Bronze  
**Source** : silver.trays — PMAF-C012501 / 2026-05-15  
**Format cible** : `{iothub}/raw/{YYYY}/{MM}/{DD}/{HH}_{mm}_{partition}.json`  
**Trigger simulé** : `pmaf_trig = "Tray"` (plateaux uniquement)

In [1]:
import os
import json
import base64
import hashlib
from datetime import datetime, timezone
from pathlib import Path
from collections import defaultdict

import pandas as pd
from deltalake import DeltaTable
from dotenv import load_dotenv

load_dotenv()
print("OK")

OK


In [2]:
STORAGE_ACCOUNT = os.getenv("SILVER_STORAGE_ACCOUNT_NAME")
STORAGE_KEY      = os.getenv("SILVER_STORAGE_ACCOUNT_KEY")
CONTAINER        = os.getenv("SILVER_CONTAINER", "silver")
TABLE_PATH       = os.getenv("SILVER_TABLE_PATH", "trays")

TABLE_URI = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/{TABLE_PATH}"

STORAGE_OPTIONS = {
    "account_name": STORAGE_ACCOUNT,
    "account_key":  STORAGE_KEY,
}

MACHINE_ID   = "PMAF-C012501"
CANDLED_DATE = "2026-05-15"

dt = DeltaTable(TABLE_URI, storage_options=STORAGE_OPTIONS)
df = dt.to_pandas(
    partitions=[
        ("machine_id",   "=", MACHINE_ID),
        ("candled_date", "=", CANDLED_DATE),
    ]
)

# Trier par timestamp d'ingestion pour respecter l'ordre chronologique
df = df.sort_values("candled_at").reset_index(drop=True)

print(f"Lignes chargées : {len(df)}")
print(f"Période : {df['candled_at'].min()} → {df['candled_at'].max()}")
df.head(3)

Lignes chargées : 1344
Période : 2026-05-15 04:32:48.638548+00:00 → 2026-05-15 07:57:28.697491+00:00


,tray_id,machine_id,candled_at,candled_date,flock,trolley,tray_seq,flock_name,trolley_name,caliber,...,n_early_dead,n_late_dead,n_missing,is_count_consistent,matrix_compact,light_flat,alarm_emergency_stop,alarm_air_pressure_fault,processed_at,bronze_path
0,74115597938121eb420596171058e96bbe78184f5eaf0a...,PMAF-C012501,2026-05-15 04:32:48.638548+00:00,2026-05-15,1,20713024,1,43e6ab99aeea0b2d6ca,20713024,1,...,2,4,0,True,1112114411111111111111111113311111131111111111...,"[135117, 136189, 135536, 0, 185909, 184676, 14...",0,0,2026-06-03 09:57:02.303577+00:00,raw/trolley/year=2026/month=05/day=15/PMAF-C01...
1,5258a070b41238f9c286d6c9ad5d12b85e537cfccd9599...,PMAF-C012501,2026-05-15 04:32:54.357327+00:00,2026-05-15,1,20713024,2,43e6ab99aeea0b2d6ca,20713024,1,...,0,2,0,True,1131111111111113111111131111131111111111311111...,"[184859, 185531, 0, 185327, 185530, 185469, 18...",0,0,2026-06-03 09:57:02.303577+00:00,raw/trolley/year=2026/month=05/day=15/PMAF-C01...
2,77ae7894195d687ce8d145c09c537d0d4318738d529d3c...,PMAF-C012501,2026-05-15 04:33:00.574953+00:00,2026-05-15,1,20713024,3,43e6ab99aeea0b2d6ca,20713024,1,...,1,1,0,True,1111111111111111111113111111111111111111111111...,"[146448, 173058, 185251, 135635, 185720, 18561...",0,0,2026-06-03 09:57:02.303577+00:00,raw/trolley/year=2026/month=05/day=15/PMAF-C01...


In [3]:
def matrix_compact_to_tags(matrix_compact: str) -> list[dict]:
    """
    Reconstruit les 150 tags final_candled_eggs[i][j]
    depuis la chaîne compacte de 150 caractères (ordre ligne-major 15x10)
    """
    tags = []
    for idx, val in enumerate(matrix_compact):
        row = (idx // 10) + 1   # 1-based
        col = (idx % 10) + 1    # 1-based
        tags.append({
            "id": f"LaserLife.{MACHINE_ID}.final_candled_eggs[{row}][{col}]",
            "value": int(val),
            "quality": 192,
            "timestamp": None  # rempli plus bas
        })
    return tags


def light_flat_to_tags(light_flat: list, machine_id: str) -> list[dict]:
    """
    Reconstruit les 150 tags laser1_light_transmitted_eggs[i][j]
    depuis le tableau plat light_flat (ordre ligne-major 15x10)
    """
    tags = []
    if light_flat is None:
        return tags
    for idx, val in enumerate(light_flat):
        row = (idx // 10) + 1
        col = (idx % 10) + 1
        tags.append({
            "id": f"LaserLife.{machine_id}.laser1_light_transmitted_eggs[{row}][{col}]",
            "value": int(val) if val is not None else 0,
            "quality": 192,
            "timestamp": None
        })
    return tags


def row_to_iot_body(row: pd.Series) -> dict:
    """
    Reconstruit le body IoT Hub complet depuis une ligne Silver
    """
    ts = row["candled_at"]
    if hasattr(ts, "isoformat"):
        ts_str = ts.isoformat()
    else:
        ts_str = str(ts)

    # Tags scalaires
    scalar_tags = [
        {"id": f"LaserLife.{row['machine_id']}.pmaf_trig",                   "value": "Tray",                       "quality": 192, "timestamp": ts_str},
        {"id": f"LaserLife.{row['machine_id']}.flock_number",                 "value": int(row["flock"]),             "quality": 192, "timestamp": ts_str},
        {"id": f"LaserLife.{row['machine_id']}.flock_name",                   "value": str(row["flock_name"] or ""), "quality": 192, "timestamp": ts_str},
        {"id": f"LaserLife.{row['machine_id']}.trolley_name",                 "value": str(row["trolley_name"] or row["trolley"]), "quality": 192, "timestamp": ts_str},
        {"id": f"LaserLife.{row['machine_id']}.setter_tray_number_flock",     "value": int(row["setter_tray_number_flock"] or 0), "quality": 192, "timestamp": ts_str},
        {"id": f"LaserLife.{row['machine_id']}.caliber",                      "value": str(row["caliber"] or ""),    "quality": 192, "timestamp": ts_str},
        {"id": f"LaserLife.{row['machine_id']}.hour",                         "value": ts.hour,                      "quality": 192, "timestamp": ts_str},
        {"id": f"LaserLife.{row['machine_id']}.minute",                       "value": ts.minute,                    "quality": 192, "timestamp": ts_str},
        {"id": f"LaserLife.{row['machine_id']}.second",                       "value": ts.second,                    "quality": 192, "timestamp": ts_str},
        {"id": f"LaserLife.{row['machine_id']}.alarm_emergency_stop",         "value": int(row["alarm_emergency_stop"] or 0), "quality": 192, "timestamp": ts_str},
        {"id": f"LaserLife.{row['machine_id']}.alarm_air_pressure_fault",     "value": int(row["alarm_air_pressure_fault"] or 0), "quality": 192, "timestamp": ts_str},
        {"id": f"LaserLife.{row['machine_id']}.#_egg_flock",                  "value": int(row["n_total"]),           "quality": 192, "timestamp": ts_str},
        {"id": f"LaserLife.{row['machine_id']}.#_living_embryo_flock",        "value": int(row["n_fertile"]),         "quality": 192, "timestamp": ts_str},
        {"id": f"LaserLife.{row['machine_id']}.#_early_dead_egg_flock",       "value": int(row["n_early_dead"]),      "quality": 192, "timestamp": ts_str},
        {"id": f"LaserLife.{row['machine_id']}.#_dead_and_rotten_flock",      "value": int(row["n_late_dead"]),       "quality": 192, "timestamp": ts_str},
        {"id": f"LaserLife.{row['machine_id']}.#_missing_egg_flock",          "value": int(row["n_missing"]),         "quality": 192, "timestamp": ts_str},
    ]

    # Tags matrice (150 valeurs)
    matrix_tags = matrix_compact_to_tags(row["matrix_compact"])
    for t in matrix_tags:
        t["timestamp"] = ts_str

    # Tags lumière (150 valeurs)
    light_tags = light_flat_to_tags(row["light_flat"], row["machine_id"])
    for t in light_tags:
        t["timestamp"] = ts_str

    body = {
        "process_serial_number": row["machine_id"],
        "box": "LaserLife",
        "values": scalar_tags + matrix_tags + light_tags
    }

    return body


print("Fonctions de reconstruction définies.")

Fonctions de reconstruction définies.


In [4]:
def assign_batch(candled_at, batch_seconds: int = 60) -> str:
    """
    Simule le timestamp d'écriture IoT Hub (HH_MM)
    En prod c'est le moment où IoT Hub écrit le fichier,
    pas le timestamp des données à l'intérieur
    """
    epoch  = int(candled_at.timestamp())
    bucket = (epoch // batch_seconds) * batch_seconds
    dt_bucket = datetime.fromtimestamp(bucket, tz=timezone.utc)
    return dt_bucket.strftime("%H_%M")

df["batch_key"]    = df["candled_at"].apply(assign_batch)
df["enqueued_at"]  = df["candled_at"].apply(
    lambda ts: datetime.fromtimestamp(
        (int(ts.timestamp()) // 60) * 60, tz=timezone.utc
    )
)

batch_counts = df.groupby("batch_key").size()
print(f"Nombre de batches simulés : {len(batch_counts)}")
print(f"Plateaux par batch (min/moy/max) : "
      f"{batch_counts.min()} / {batch_counts.mean():.1f} / {batch_counts.max()}")
batch_counts.head(10)

Nombre de batches simulés : 180
Plateaux par batch (min/moy/max) : 1 / 7.5 / 11


batch_key
04_32     2
04_33    11
04_34     9
04_35     6
04_36     4
04_37     9
04_38    10
04_39    10
04_40     9
04_41     9
dtype: int64

In [5]:
IOTHUB_NAME = "iothub-ecat-candling-dev"
OUTPUT_DIR  = Path("../data/ndjson_sim")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Nettoyer les anciens fichiers
for f in OUTPUT_DIR.glob("**/*.json"):
    f.unlink()

files_generated = []

for batch_key, group in df.groupby("batch_key"):
    messages     = []
    ts_write     = group["enqueued_at"].iloc[0]  # timestamp d'écriture IoT Hub

    year      = ts_write.strftime("%Y")
    month     = ts_write.strftime("%m")
    day       = ts_write.strftime("%d")
    partition = 0

    sub_dir  = OUTPUT_DIR / IOTHUB_NAME / "raw" / year / month / day
    sub_dir.mkdir(parents=True, exist_ok=True)
    filepath = sub_dir / f"{batch_key}_{partition}.json"

    for _, row in group.iterrows():
        body_dict = row_to_iot_body(row)
        body_b64  = base64.b64encode(
            json.dumps(body_dict, ensure_ascii=False).encode("utf-8")
        ).decode("utf-8")

        enqueued = ts_write.strftime("%Y-%m-%dT%H:%M:%S.%f") + "Z"

        message = {
            "EnqueuedTimeUtc": enqueued,
            "SystemProperties": {
                "connectionDeviceId": f"device-{row['machine_id'].lower()}",
                "enqueuedTime":       enqueued,
            },
            "Properties": {},
            "Body": body_b64
        }
        messages.append(json.dumps(message, ensure_ascii=False))

    with open(filepath, "w", encoding="utf-8") as f:
        f.write("\n".join(messages))

    files_generated.append({
        "file":     str(filepath.relative_to(OUTPUT_DIR)),
        "batch":    batch_key,
        "messages": len(messages),
        "written_at": ts_write.isoformat(),
    })

summary = pd.DataFrame(files_generated)
print(f"{len(files_generated)} fichiers générés")
print(f"Structure : {IOTHUB_NAME}/raw/YYYY/MM/DD/HH_MM_0.json")
summary.head(5)

180 fichiers générés
Structure : iothub-ecat-candling-dev/raw/YYYY/MM/DD/HH_MM_0.json


,file,batch,messages,written_at
0,iothub-ecat-candling-dev\raw\2026\05\15\04_32_...,04_32,2,2026-05-15T04:32:00+00:00
1,iothub-ecat-candling-dev\raw\2026\05\15\04_33_...,04_33,11,2026-05-15T04:33:00+00:00
2,iothub-ecat-candling-dev\raw\2026\05\15\04_34_...,04_34,9,2026-05-15T04:34:00+00:00
3,iothub-ecat-candling-dev\raw\2026\05\15\04_35_...,04_35,6,2026-05-15T04:35:00+00:00
4,iothub-ecat-candling-dev\raw\2026\05\15\04_36_...,04_36,4,2026-05-15T04:36:00+00:00


In [9]:
# Vérifier que le premier fichier est bien décodable
all_files = sorted(OUTPUT_DIR.glob("**/*.json"))
print(f"Fichiers générés : {len(all_files)}")
print(f"Structure exemple : {all_files[0].relative_to(OUTPUT_DIR)}")

sample_file = all_files[0]
print(f"\nVérification : {sample_file.name}\n")

with open(sample_file, "r", encoding="utf-8") as f:
    lines = [l.strip() for l in f if l.strip()]

print(f"Messages dans ce fichier : {len(lines)}")

first = json.loads(lines[0])
body  = json.loads(base64.b64decode(first["Body"]).decode("utf-8"))

print(f"\nmachine_id  : {body['process_serial_number']}")
print(f"box         : {body['box']}")
print(f"nb tags     : {len(body['values'])}")
print(f"enqueued_at : {first['EnqueuedTimeUtc']}")

key_tags = {
    v["id"].split(".")[-1]: v["value"]
    for v in body["values"]
    if not v["id"].split(".")[-1].startswith(("final_", "laser1_"))
}
print("\nTags scalaires :")
for k, v in key_tags.items():
    print(f"  {k:<40} = {v}")

Fichiers générés : 180
Structure exemple : iothub-ecat-candling-dev\raw\2026\05\15\04_32_0.json

Vérification : 04_32_0.json

Messages dans ce fichier : 2

machine_id  : PMAF-C012501
box         : LaserLife
nb tags     : 316
enqueued_at : 2026-05-15T04:32:00.000000Z

Tags scalaires :
  pmaf_trig                                = Tray
  flock_number                             = 1
  flock_name                               = 43e6ab99aeea0b2d6ca
  trolley_name                             = 20713024
  setter_tray_number_flock                 = 1
  caliber                                  = 1
  hour                                     = 4
  minute                                   = 32
  second                                   = 48
  alarm_emergency_stop                     = 0
  alarm_air_pressure_fault                 = 0
  #_egg_flock                              = 150
  #_living_embryo_flock                    = 130
  #_early_dead_egg_flock                   = 2
  #_dead_and_rotten_flo